In [25]:
import pandas as pd
import sys
from pathlib import Path
import re

project_root = Path.cwd().parent
sys.path.append(str(project_root))

## Utility

In [59]:
def export_csv(df: pd.DataFrame, filename: str):
    df.to_csv(project_root / "data" / "clean" / filename, index=False)

## Movies

In [33]:
movies = pd.read_csv(project_root / "data" / "raw" / "movies.csv")
movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


### Separate title and year

In [34]:
def separate_title(text):
    match = re.match(r"^(.*)\s\((\d{4})\)\s*$", text.strip())
    return match.group(1), int(match.group(2))

In [35]:
movies[['title', 'year']] = movies['title'].str.extract(r'^(.*)\s\((\d{4})\)\s*$')
movies['year'] = movies['year'].astype('Int64') 
movies

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995
...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,Action|Animation|Comedy|Fantasy,2017
9738,193583,No Game No Life: Zero,Animation|Comedy|Fantasy,2017
9739,193585,Flint,Drama,2017
9740,193587,Bungo Stray Dogs: Dead Apple,Action|Animation,2018


### Change column name

In [36]:
movies.rename(columns={'movieId': 'movielens_id'}, inplace=True)
movies

,movielens_id,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995
...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,Action|Animation|Comedy|Fantasy,2017
9738,193583,No Game No Life: Zero,Animation|Comedy|Fantasy,2017
9739,193585,Flint,Drama,2017
9740,193587,Bungo Stray Dogs: Dead Apple,Action|Animation,2018


### Drop column: genres

In [37]:
movies.drop(columns='genres', inplace=True)
movies

,movielens_id,title,year
0,1,Toy Story,1995
1,2,Jumanji,1995
2,3,Grumpier Old Men,1995
3,4,Waiting to Exhale,1995
4,5,Father of the Bride Part II,1995
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,2017
9738,193583,No Game No Life: Zero,2017
9739,193585,Flint,2017
9740,193587,Bungo Stray Dogs: Dead Apple,2018


In [71]:
movies = movies[movies.title.notnull()]
movies.info()

<class 'pandas.DataFrame'>
Index: 9729 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   movielens_id  9729 non-null   int64
 1   title         9729 non-null   str  
 2   year          9729 non-null   Int64
dtypes: Int64(1), int64(1), str(1)
memory usage: 313.5 KB


### Export

In [ ]:
# export_csv(movies, "movies.csv")

## links

In [14]:
links = pd.read_csv(project_root / "data" / "raw" / "links.csv")
links

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
9737,193581,5476944,432131.0
9738,193583,5914996,445030.0
9739,193585,6397426,479308.0
9740,193587,8391976,483455.0


In [17]:
links.rename(columns={ 'movieId': 'movielens_id',  'imdbId': 'imdb_id', 'tmdbId': 'tmdb_id'}, inplace=True)
links

,movielens_id,imdb_id,tmdb_id
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
9737,193581,5476944,432131.0
9738,193583,5914996,445030.0
9739,193585,6397426,479308.0
9740,193587,8391976,483455.0


In [19]:
links.tmdb_id = links.tmdb_id.astype('Int64') 
links

,movielens_id,imdb_id,tmdb_id
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862
...,...,...,...
9737,193581,5476944,432131
9738,193583,5914996,445030
9739,193585,6397426,479308
9740,193587,8391976,483455


### Check null and doublons

In [40]:
links = links[links.tmdb_id.notnull()]
links[links.tmdb_id.isnull()]

,movielens_id,imdb_id,tmdb_id


In [49]:
links = links[~links.tmdb_id.duplicated()]

In [50]:
links[links.tmdb_id.duplicated()]

,movielens_id,imdb_id,tmdb_id


In [ ]:
# export_csv(links, "links.csv")

## ratings

In [21]:
ratings = pd.read_csv(project_root / "data" / "raw" / "ratings.csv")
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [23]:
ratings.rename(columns={ 'movieId': 'movielens_id', 'userId': 'user_id' }, inplace=True)
ratings

,user_id,movielens_id,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [63]:
ratings.drop(columns='timestamp', inplace=True)
ratings

,user_id,movielens_id,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [ ]:
# export_csv(ratings, "ratings.csv")

## Genres

In [26]:
from src.TMDBExtractor import TMDBExtractor

In [27]:
extractor = TMDBExtractor()
extractor.genre_mapping

{28: 'Action',
 12: 'Adventure',
 16: 'Animation',
 35: 'Comedy',
 80: 'Crime',
 99: 'Documentary',
 18: 'Drama',
 10751: 'Family',
 14: 'Fantasy',
 36: 'History',
 27: 'Horror',
 10402: 'Music',
 9648: 'Mystery',
 10749: 'Romance',
 878: 'Science Fiction',
 10770: 'TV Movie',
 53: 'Thriller',
 10752: 'War',
 37: 'Western'}

In [30]:
mapping = extractor.genre_mapping

genres = pd.DataFrame({
    'genre_id': mapping.keys(),
    'genre_name': mapping.values()
})
genres

,genre_id,genre_name
0,28,Action
1,12,Adventure
2,16,Animation
3,35,Comedy
4,80,Crime
5,99,Documentary
6,18,Drama
7,10751,Family
8,14,Fantasy
9,36,History


In [ ]:
# export_csv(genres, "genres.csv")